# TuRBO / Trust-Region BO Workflow

TuRBOはsurrogate modelやoptimizer backendそのものではなく、**stateful search geometry**です。

```text
completed data → surrogate → acquisition
                              ↓
global bounds + TuRBO state → local trust-region bounds
                              ↓
                       acquisition optimizer
                              ↓
                           candidate
                              ↓
                    objective evaluation
                              ↓
                         update_state()
                              ↓
                center / counters / length
```

このNotebookではbaseline TuRBO-1 loop、Thompson candidate generation、restart semanticsを実行します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import SingleTaskGP
from robotorchan.optim.trust_region import TuRBOState, TuRBOStrategy

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Problem and initial observations

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    target = torch.tensor([0.72, 0.35, 0.58])
    return -((X - target) ** 2).sum(dim=-1, keepdim=True)


dim = 3
bounds = torch.stack([torch.zeros(dim), torch.ones(dim)])
train_X = torch.rand(10, dim)
train_Y = objective(train_X)
best_index = train_Y.squeeze(-1).argmax()
incumbent = train_X[best_index]
initial_best = float(train_Y[best_index].item())

print("initial incumbent:", incumbent)
print("initial best value:", initial_best)

## 3. Initialize TuRBO state

`center`とstateはpublic/original input coordinatesで保持します。`length`はglobal bound幅に対する相対的なlocalization scaleです。

In [ ]:
state = TuRBOState(dim=dim, best_value=initial_best)
strategy = TuRBOStrategy(
    bounds,
    center=incumbent,
    state=state,
    num_restarts=3,
    raw_samples=48,
    seed=17,
)

trust_bounds = strategy.trust_region_bounds()
print("initial length:", strategy.state.length)
print("trust-region bounds:\n", trust_bounds)

## 4. One TuRBO candidate

surrogate/acquisitionは通常どおり構築し、TuRBOは現在のlocal bounds内でacquisition optimizationを実行します。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
result = strategy.optimize(acquisition, q=1)
candidate = result.candidates.detach()
local_bounds = result.metadata["trust_region_bounds"]

assert candidate.shape == torch.Size([1, dim])
assert torch.all(candidate >= local_bounds[0])
assert torch.all(candidate <= local_bounds[1])
print("candidate:", candidate)
print("local bounds:\n", local_bounds)

## 5. Evaluation and state transition

objective評価はstrategyの責務ではありません。完了した観測だけを`update_state()`へ戻します。

In [ ]:
new_Y = objective(candidate)
old_center = strategy.center.clone()
old_length = strategy.state.length
strategy.update_state(new_Y, candidates=candidate)

train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, new_Y], dim=0)

print("observed value:", new_Y.item())
print("old center:", old_center)
print("new center:", strategy.center)
print("length:", old_length, "->", strategy.state.length)
print("success counter:", strategy.state.success_counter)
print("failure counter:", strategy.state.failure_counter)

## 6. Baseline TuRBO-1 loop

各iterationでmodelをcompleted observationsからrefitし、local acquisition optimization後にstateを1回更新します。

In [ ]:
history = []
for iteration in range(3):
    model = SingleTaskGP(train_X, train_Y)
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
    )
    result = strategy.optimize(acquisition, q=1)
    new_X = result.candidates.detach()
    new_Y = objective(new_X)
    strategy.update_state(new_Y, candidates=new_X)
    train_X = torch.cat([train_X, new_X], dim=0)
    train_Y = torch.cat([train_Y, new_Y], dim=0)
    history.append(
        (
            iteration,
            float(new_Y.item()),
            strategy.state.length,
            strategy.state.success_counter,
            strategy.state.failure_counter,
        )
    )

for row in history:
    print(row)

## 7. Thompson sampling is another candidate-generation path

TuRBOはlocal candidate poolを作り、posterior samplingによるThompson selectionも利用できます。これはgradient-based acquisition optimizationとは別経路です。

In [ ]:
ts_model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(ts_model.make_mll())
ts_model.eval()
ts_result = strategy.thompson_sample(
    ts_model,
    q=1,
    n_candidates=128,
)
ts_candidate = ts_result.candidates
ts_bounds = ts_result.metadata["trust_region_bounds"]

assert ts_result.acquisition_value is None
assert ts_result.metadata["candidate_generation"] == "thompson"
assert torch.all(ts_candidate >= ts_bounds[0])
assert torch.all(ts_candidate <= ts_bounds[1])
print("Thompson candidate:", ts_candidate)

## 8. Restart is explicit

`length < length_min`でrestart flagが立っても、暗黙にglobal searchへ戻りません。`restart()`を明示的に呼び、local stateをresetします。

In [ ]:
restart_state = TuRBOState(
    dim=dim,
    length=0.1,
    length_min=0.1,
    best_value=strategy.state.best_value,
    restart_triggered=True,
)
restart_strategy = TuRBOStrategy(
    bounds,
    center=strategy.center,
    state=restart_state,
    seed=29,
)
restart_strategy.restart()

assert not restart_strategy.state.restart_triggered
assert restart_strategy.state.restart_count == 1
print("restart center:", restart_strategy.center)
print("restart length:", restart_strategy.state.length)
print("restart count:", restart_strategy.state.restart_count)

## 9. TuRBO owns geometry, not the whole BO stack

| Responsibility | Owner |
| --- | --- |
| surrogate/posterior | model |
| acquisition semantics | acquisition |
| local search bounds | TuRBO strategy |
| numerical optimization | selected optimizer backend |
| objective evaluation | caller / experiment |
| completed-result state transition | TuRBO strategy |
| unresolved evaluations | acquisition `X_pending` |
| model fantasies | model/acquisition |

したがってTuRBOを新しいsurrogateや新しいoptimizer backendとして扱いません。

## 10. Important boundaries

- `center`、candidate、boundsはpublic/original input spaceです。
- model-owned PCA/PLS等のlatent coordinatesへtrust regionを暗黙に移しません。
- ARD weightを使う場合もpublic dimensionsとの対応を確認します。
- batch TuRBOでは`state.batch_size`とcandidate generationの`q`を一致させます。
- asyncではpending evaluationでstateを更新せず、completed resultだけを更新します。
- input perturbation scenario axisはTuRBOの`q`やdimensionへ追加しません。
- multi-objective acquisitionでもTuRBO-1 state updateにはcandidateごとのscalar state utilityが必要です。
- mixed TuRBOはcontinuous dimensionsだけを縮小し、categorical値へ人工的な数値距離を導入しません。
- multi-fidelityではdesign dimensionsのlocalizationとfidelity selectionを同一視しません。

## 11. Related documentation

- `docs/optimization/turbo.md`
- `docs/optimization/high_dimensional_search.md`
- `docs/theory/optimization/09_trust_region.md`
- `31_acquisition_optimization.ipynb`
- `33_acquisition_initialization.ipynb`

TuRBOの中心は、局所領域を状態として管理しながら既存のmodel・acquisition・optimizerを再利用することです。